# RAG Pipeline — From Scratch
## Notebook 4 — AI Engineer Practical Series

RAG (Retrieval-Augmented Generation) solves the core problem of LLMs:
they can't know what's in YOUR documents.

### How it works:
1. **Chunk** — split documents into small pieces
2. **Embed** — convert each chunk into a vector (numbers representing meaning)
3. **Store** — save vectors in a vector database
4. **Retrieve** — find the most relevant chunks for a query
5. **Generate** — send retrieved chunks + question to the LLM

### What we build:
A RAG pipeline from scratch — no LangChain, no abstractions.
Understanding the internals before using the frameworks.

YOUR QUESTION
     │
     ▼
[Embed question] ← sentence-transformers
     │
     ▼
[Vector similarity search]
     │
     ├── Doc 1: similarity = 0.91 ✅
     ├── Doc 2: similarity = 0.45
     ├── Doc 3: similarity = 0.87 ✅
     └── Doc 4: similarity = 0.12
     │
     ▼
[Top 2 most relevant docs retrieved]
     │
     ▼
[LLM prompt] ← Groq
  "Answer using these docs:
   Doc 1: ...
   Doc 3: ...
   Question: ..."
     │
     ▼
ACCURATE ANSWER ✅

In [ ]:
# Install required packages
# Run this cell once, then restart kernel

%pip install groq
%pip install sentence-transformers
%pip install tf-keras
%pip install numpy
%pip install python-dotenv

In [1]:
import os
import numpy as np
from dotenv import load_dotenv
from groq import Groq
from sentence_transformers import SentenceTransformer

load_dotenv()

# Groq for generation
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = os.getenv("GROQ_MODEL")

# Local embedding model — no API needed
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

def get_embedding(text):
    return embed_model.encode(text)

# Test
test = get_embedding("hello world")
print(f"Groq client ready ✅")
print(f"Embedding model: all-MiniLM-L6-v2 (local)")
print(f"Embedding dimensions: {test.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\adars\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\adars\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Groq client ready ✅
Embedding model: all-MiniLM-L6-v2 (local)
Embedding dimensions: (384,)


## Step 1: Load Documents
Our knowledge base — in production this would be PDFs, Word docs, databases.

In [2]:
documents = [
    "RAG stands for Retrieval-Augmented Generation. It combines information retrieval with text generation to produce accurate, grounded responses.",
    "LangChain is a framework for building LLM applications. It provides tools for chaining LLM calls, memory management, and agent creation.",
    "Vector databases store embeddings — numerical representations of text. They enable semantic search by finding similar vectors using cosine similarity.",
    "Fine-tuning trains a pre-existing model on new data to adapt its behaviour. LoRA is a popular fine-tuning technique that only trains small adapter matrices.",
    "Prompt engineering is the practice of designing inputs to get reliable outputs from LLMs. Key techniques include few-shot prompting, chain of thought, and structured outputs.",
    "FastAPI is a modern Python web framework for building APIs. It is the standard for serving ML models in production due to its speed and automatic documentation.",
    "Docker packages applications and their dependencies into containers. This ensures the application runs consistently across different environments.",
    "LangSmith is an observability platform for LLM applications. It provides tracing, evaluation, and monitoring for LangChain-based systems.",
]

print(f"Knowledge base loaded ✅ — {len(documents)} documents")
for i, doc in enumerate(documents):
    print(f"[{i}] {doc[:80]}...")

Knowledge base loaded ✅ — 8 documents
[0] RAG stands for Retrieval-Augmented Generation. It combines information retrieval...
[1] LangChain is a framework for building LLM applications. It provides tools for ch...
[2] Vector databases store embeddings — numerical representations of text. They enab...
[3] Fine-tuning trains a pre-existing model on new data to adapt its behaviour. LoRA...
[4] Prompt engineering is the practice of designing inputs to get reliable outputs f...
[5] FastAPI is a modern Python web framework for building APIs. It is the standard f...
[6] Docker packages applications and their dependencies into containers. This ensure...
[7] LangSmith is an observability platform for LLM applications. It provides tracing...


## Step 2: Embed Documents
Convert each document into a vector.
Similar documents will have similar vectors.

In [ ]:
print("Embedding documents...")
doc_embeddings = []

for i, doc in enumerate(documents):
    embedding = get_embedding(doc)
    doc_embeddings.append(embedding)
    print(f"  [{i}] embedded ✅ — shape: {embedding.shape}")

doc_embeddings = np.array(doc_embeddings)
print(f"\nAll documents embedded ✅")
print(f"Embeddings matrix shape: {doc_embeddings.shape}")
print(f"First embedding vector(truncated) : {[f'{x:.4f}' for x in doc_embeddings[0][:10]]}")

Embedding documents...
  [0] embedded ✅ — shape: (384,)
  [1] embedded ✅ — shape: (384,)
  [2] embedded ✅ — shape: (384,)
  [3] embedded ✅ — shape: (384,)
  [4] embedded ✅ — shape: (384,)
  [5] embedded ✅ — shape: (384,)
  [6] embedded ✅ — shape: (384,)
  [7] embedded ✅ — shape: (384,)

All documents embedded ✅
Embeddings matrix shape: (8, 384)
First embedding vector : ['-0.0751', '0.0607', '0.0152', '0.0068', '-0.0744', '0.0364', '0.0388', '0.0286', '0.0187', '-0.0555']


## Step 3: Retrieve
Find the most relevant documents for a query using cosine similarity.
Cosine similarity measures the angle between two vectors — closer to 1 = more similar.

In [15]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve(query, top_k=2):
    query_embedding = get_embedding(query)
    
    # Score every document
    scores = []
    for i, doc_embedding in enumerate(doc_embeddings):
        score = cosine_similarity(query_embedding, doc_embedding)
        scores.append((score, i, documents[i]))
    
    # Sort by score descending
    scores.sort(reverse=True)
    
    print(f"Query: {query}\n")
    print("Scores:")
    for score, idx, doc in scores:
        print(f"  [{idx}] {score:.4f} — {doc[:60]}...")
    
    # Return top k
    return [doc for _, _, doc in scores[:top_k]]

# Test retrieval
results = retrieve("How does RAG work?")
print(f"\nTop 2 retrieved:\n")
for r in results:
    print(f"→ {r}\n")

Query: How does RAG work?

Scores:
  [0] 0.6046 — RAG stands for Retrieval-Augmented Generation. It combines i...
  [1] 0.1488 — LangChain is a framework for building LLM applications. It p...
  [6] 0.1143 — Docker packages applications and their dependencies into con...
  [7] 0.1137 — LangSmith is an observability platform for LLM applications....
  [5] 0.0955 — FastAPI is a modern Python web framework for building APIs. ...
  [3] 0.0801 — Fine-tuning trains a pre-existing model on new data to adapt...
  [2] 0.0133 — Vector databases store embeddings — numerical representation...
  [4] -0.0643 — Prompt engineering is the practice of designing inputs to ge...

Top 2 retrieved:

→ RAG stands for Retrieval-Augmented Generation. It combines information retrieval with text generation to produce accurate, grounded responses.

→ LangChain is a framework for building LLM applications. It provides tools for chaining LLM calls, memory management, and agent creation.



## Step 4: Generate
Send retrieved chunks + question to the LLM and generate a grounded answer.

In [16]:
def generate(query, retrieved_docs):
    context = "\n\n".join([f"Doc {i+1}: {doc}" 
                           for i, doc in enumerate(retrieved_docs)])
    
    response = groq_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": """You are a precise AI assistant.
Answer the question using ONLY the provided context.
If the answer isn't in the context, say 'Not found in knowledge base.'
Always cite which Doc number you used."""},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
        ]
    )
    return response.choices[0].message.content

# Test generation
query = "How does RAG work?"
retrieved = retrieve(query)
answer = generate(query, retrieved)
print(f"\nAnswer:\n{answer}")

Query: How does RAG work?

Scores:
  [0] 0.6046 — RAG stands for Retrieval-Augmented Generation. It combines i...
  [1] 0.1488 — LangChain is a framework for building LLM applications. It p...
  [6] 0.1143 — Docker packages applications and their dependencies into con...
  [7] 0.1137 — LangSmith is an observability platform for LLM applications....
  [5] 0.0955 — FastAPI is a modern Python web framework for building APIs. ...
  [3] 0.0801 — Fine-tuning trains a pre-existing model on new data to adapt...
  [2] 0.0133 — Vector databases store embeddings — numerical representation...
  [4] -0.0643 — Prompt engineering is the practice of designing inputs to ge...

Answer:
RAG stands for Retrieval-Augmented Generation. It combines information retrieval with text generation to produce accurate, grounded responses. (Doc 1)


## Step 5: Full RAG Pipeline
Putting it all together — chunk, embed, retrieve, generate in one function.

In [17]:
def rag(query, top_k=2):
    print(f"Question: {query}")
    print("─" * 50)
    
    # Retrieve
    retrieved_docs = retrieve(query, top_k)
    
    # Generate
    answer = generate(query, retrieved_docs)
    
    print(f"\nAnswer: {answer}")
    print("─" * 50)
    return answer

# Test with multiple questions
rag("What is LangChain used for?")
rag("How does Docker help developers?")
rag("What is the capital of France?")  # not in knowledge base

Question: What is LangChain used for?
──────────────────────────────────────────────────
Query: What is LangChain used for?

Scores:
  [1] 0.6773 — LangChain is a framework for building LLM applications. It p...
  [7] 0.4798 — LangSmith is an observability platform for LLM applications....
  [6] 0.2646 — Docker packages applications and their dependencies into con...
  [0] 0.2634 — RAG stands for Retrieval-Augmented Generation. It combines i...
  [5] 0.1476 — FastAPI is a modern Python web framework for building APIs. ...
  [2] 0.0697 — Vector databases store embeddings — numerical representation...
  [4] 0.0624 — Prompt engineering is the practice of designing inputs to ge...
  [3] 0.0607 — Fine-tuning trains a pre-existing model on new data to adapt...

Answer: LangChain is used for building LLM (Language Model) applications. (Doc 1)
──────────────────────────────────────────────────
Question: How does Docker help developers?
──────────────────────────────────────────────────
Query: 

'Not found in knowledge base.'

## Combined RAG Implementation with a text corpus

In [71]:
import os
import numpy as np
from groq import Groq
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv()

# Initialize clients
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))  
MODEL = os.getenv("GROQ_MODEL", "mixtral-8x7b-32768")  # Default fallback

# Local embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

text_corpus = [
    "Artificial Intelligence (AI) is the simulation of human intelligence in machines",
    "Machine Learning (ML) is a subset of AI that enables systems to learn from data",
    "Deep Learning uses neural networks with multiple layers to process complex patterns",
    "Natural Language Processing (NLP) allows AI to understand and generate human language",
    "Computer Vision enables AI to interpret and analyze visual information from images and videos",
    "Reinforcement Learning trains AI through rewards and punishments like teaching a dog tricks",
    "Generative AI can create new content including text, images, music, and code",
    "Large Language Models (LLMs) like GPT are trained on massive text datasets",
    "AI systems require large amounts of quality data to perform effectively",
    "Bias in AI training data can lead to unfair or discriminatory outcomes",
    "Explainable AI (XAI) focuses on making AI decisions transparent and interpretable",
    "Edge AI runs machine learning models directly on devices without cloud connectivity",
    "Transfer Learning allows AI models to apply knowledge from one task to another",
    "Federated Learning trains AI across decentralized devices without sharing raw data",
    "AI hallucinations occur when models generate confident but incorrect information",
    "Prompt engineering is the practice of designing effective inputs for AI models",
    "Multimodal AI can process and integrate multiple types of data (text, image, audio)",
    "AI agents are autonomous systems that perceive environments and take actions to achieve goals",
    "Retrieval-Augmented Generation (RAG) combines LLMs with external knowledge bases",
    "AI alignment ensures artificial intelligence systems pursue goals that benefit humanity"
]

# Create embeddings
def transform_text(text):
    return embed_model.encode(text)

def create_embeddings(text_corpus):
    vector_embeddings = []
    for text in text_corpus:
        vector_embeddings.append(transform_text(text))
    return np.array(vector_embeddings)

doc_embeddings = create_embeddings(text_corpus)

# Retrieve relevant chunks
def relevant_chunks(documents, query, top_k):
    # FIXED: Use transform_text instead of undefined get_embedding
    query_embedding = transform_text(query)
    chunks = []
    
    for i, doc in enumerate(documents):
        # Handle zero vectors to avoid division by zero
        norm_doc = np.linalg.norm(doc)
        norm_query = np.linalg.norm(query_embedding)
        
        if norm_doc == 0 or norm_query == 0:
            score = 0.0
        else:
            score = np.dot(doc, query_embedding) / (norm_doc * norm_query)
        
        # Store as (score, index, text) for easier sorting
        chunks.append((score, i, text_corpus[i]))
    
    # Sort by score descending
    chunks.sort(key=lambda x: x[0], reverse=True)
    return chunks[:top_k]

# Generate response with CONSISTENT formatting
def generate_data(query, relevant_chunks):
    # Format context with clear document references
    context_parts = []
    for score, idx, text in relevant_chunks:
        context_parts.append(f"[Doc {idx + 1}]: {text}")
    context = "\n\n".join(context_parts)
    
    # FIXED: Clear, consistent instruction for response format
    response = groq_client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an AI analyst tasked with answering questions about AI. "
                    "CRITICAL INSTRUCTIONS:\n"
                    "1. ONLY use information from the provided context\n"
                    "2. If the answer is not in the context, say: 'Unable to answer the question'\n"
                    "3. ALWAYS cite your source using the format: [Doc X] where X is the document number\n"
                    "4. Do NOT use markdown, bullet points, or follow-up questions\n"
                    "5. Answer concisely in 1-2 sentences\n"
                    "6. Format: '[your answer] (Source: [Doc X])'"
                )
            },
            {
                "role": "user",
                "content": f"Query: {query}\n\nContext:\n{context}"
            }
        ],
        temperature=0.3,  # Lower temperature for consistent responses
        max_tokens=150
    )
    
    return response.choices[0].message.content

def rag(query, top_k=2, documents=doc_embeddings):
    retrieved_chunks = relevant_chunks(documents, query, top_k=top_k)
    
    # Debug: Show what was retrieved
    print(f"\n{'='*60}")
    print(f"QUERY: {query}")
    print(f"{'='*60}")
    print(f"Retrieved {len(retrieved_chunks)} chunks:")
    for score, idx, text in retrieved_chunks:
        print(f"  → Doc {idx+1} (Score: {score:.3f}): {text[:60]}...")
    
    answer = generate_data(query, retrieved_chunks)
    
    print(f"\nANSWER: {answer}")
    print(f"{'-'*60}")

# Test questions
questions_list = [
    'What is LLM?',
    'Tell me about NLP',
    'What is my name?',  # This should trigger "Unable to answer"
    'What is RAG?'
]

# Run RAG for all questions
for question in questions_list:
    rag(question)


QUERY: What is LLM?
Retrieved 2 chunks:
  → Doc 19 (Score: 0.390): Retrieval-Augmented Generation (RAG) combines LLMs with exte...
  → Doc 8 (Score: 0.366): Large Language Models (LLMs) like GPT are trained on massive...

ANSWER: Large Language Models (LLMs) are trained on massive text datasets [Doc 8]. They are combined with external knowledge bases in models like Retrieval-Augmented Generation (RAG) [Doc 19].
------------------------------------------------------------

QUERY: Tell me about NLP
Retrieved 2 chunks:
  → Doc 4 (Score: 0.666): Natural Language Processing (NLP) allows AI to understand an...
  → Doc 8 (Score: 0.315): Large Language Models (LLMs) like GPT are trained on massive...

ANSWER: NLP enables AI to comprehend and produce human language (Source: Doc 4). It involves training models on vast text datasets, such as those used for Large Language Models like GPT (Source: Doc 8).
------------------------------------------------------------

QUERY: What is my name?
Retriev